# Notebook 7 : Multimodal Fusion QuantFormer

## Objective

This notebook extends the original QuantFormer architecture by
integrating Financial News embeddings generated using FinBERT with
Limit Order Book (LOB) features from the FI-2010 dataset.

The multimodal architecture combines:

- Limit Order Book Features
- Financial News Embeddings
- Fusion Layer
- Temporal Fusion Transformer

to improve short-term stock price movement prediction.

---

## Pipeline

Financial News
      │
      ▼
   FinBERT
      │
      ▼
News Embedding (768)

LOB Sequence
(100 × 143)
      │
      ▼
Input Projection
      │
      ▼
LSTM Encoder
      │
      ▼
Fusion Layer
      │
      ▼
Temporal Fusion Transformer
      │
      ▼
Classifier
      │
      ▼
Down / Stable / Up

In [ ]:
import os
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

print("QuantFormer Multimodal Fusion Notebook")

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"Device : {DEVICE}")


SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Random Seed : {SEED}")


PROJECT_ROOT = Path.cwd().parent

CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"

CHECKPOINT_DIR.mkdir(exist_ok=True)

print(f"Project Root : {PROJECT_ROOT}")
print(f"Checkpoint Directory : {CHECKPOINT_DIR}")
# ============================================================
# Section 2 : Load Pretrained TFT Model
# ============================================================

from src.models.tft_model import QuantFormerTFT

# ------------------------------------------------------------
# Model Configuration
# ------------------------------------------------------------

INPUT_SIZE = 143
HIDDEN_SIZE = 128
NUM_HEADS = 4
DROPOUT = 0.20
NUM_CLASSES = 3

# ------------------------------------------------------------
# Initialize Model
# ------------------------------------------------------------

tft_model = QuantFormerTFT(
    input_size=INPUT_SIZE,
    hidden_size=HIDDEN_SIZE,
    num_heads=NUM_HEADS,
    dropout=DROPOUT,
    num_classes=NUM_CLASSES
)

# ------------------------------------------------------------
# Load Trained Weights
# ------------------------------------------------------------

CHECKPOINT_PATH = (
    CHECKPOINT_DIR /
    "best_tft_model.pth"
)

print("=" * 60)
print("Loading pretrained TFT checkpoint from:")
print(CHECKPOINT_PATH)
print("=" * 60)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE
)

# ------------------------------------------------------------
# Support Both Checkpoint Formats
# ------------------------------------------------------------

if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:

    tft_model.load_state_dict(
        checkpoint["model_state_dict"]
    )

else:

    tft_model.load_state_dict(checkpoint)

# ------------------------------------------------------------
# Evaluation Mode
# ------------------------------------------------------------

tft_model = tft_model.to(DEVICE)
tft_model.eval()

print("✓ TFT Model Loaded Successfully")
print("=" * 60)

QuantFormer Multimodal Fusion Notebook
Device : cpu
Random Seed : 42
Project Root : d:\Coding\Quant Former
Checkpoint Directory : d:\Coding\Quant Former\checkpoints


c:\Users\gupta\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from src.models.tft_model import QuantFormerTFT

INPUT_SIZE = 143
HIDDEN_SIZE = 128
NUM_HEADS = 4
DROPOUT = 0.20
NUM_CLASSES = 3

# ------------------------------------------------------------
# Initialize Model
# ------------------------------------------------------------

tft_model = QuantFormerTFT(
    input_size=INPUT_SIZE,
    hidden_size=HIDDEN_SIZE,
    num_heads=NUM_HEADS,
    dropout=DROPOUT,
    num_classes=NUM_CLASSES
)

# ------------------------------------------------------------
# Load Trained Weights
# ------------------------------------------------------------

CHECKPOINT_PATH = (
    CHECKPOINT_DIR /
    "best_tft_model.pth"
)

print("=" * 60)
print("Loading pretrained TFT checkpoint from:")
print(CHECKPOINT_PATH)
print("=" * 60)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE
)

# ------------------------------------------------------------
# Support Both Checkpoint Formats
# ------------------------------------------------------------

if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:

    tft_model.load_state_dict(
        checkpoint["model_state_dict"]
    )

else:

    tft_model.load_state_dict(checkpoint)

# ------------------------------------------------------------
# Evaluation Mode
# ------------------------------------------------------------

tft_model = tft_model.to(DEVICE)
tft_model.eval()

print("✓ TFT Model Loaded Successfully")
print("=" * 60)